# G1 Locomotion -- Colab Training (Phase 2)

Run cells in order. Runtime > Change runtime type > **T4 GPU** before starting.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = '/content/drive/MyDrive/g1_checkpoints'  # survives disconnects; edit if you want a different path

In [ ]:
import os

REPO_DIR = '/content/g1_locomotion'
if os.path.isdir(REPO_DIR):
    # Re-running this cell in an already-running session (e.g. after a disconnect) must NOT
    # silently reuse a stale checkout -- `git clone` into an existing dir just fails and leaves
    # old code in place. Force it back in sync with whatever is on GitHub right now instead.
    %cd {REPO_DIR}
    !git fetch origin
    !git reset --hard origin/main
else:
    !git clone https://github.com/shubhamt2897/Humanoid_LocomotioN.git {REPO_DIR}
    %cd {REPO_DIR}

In [ ]:
# Deliberately NOT installing torch -- Colab ships a CUDA-matched build preinstalled.
# requirements.txt pins torch too (for local/CPU use); skip that line here.
!pip install -q mujoco==3.12.0 rsl-rl-lib==5.5.0 tensordict==0.14.0 wandb==0.29.0 onnx==1.22.0

import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'CUDA not available -- check Runtime > Change runtime type > T4 GPU'

In [ ]:
import wandb
wandb.login()  # paste your API key from wandb.ai/authorize when prompted, once per Colab session

In [ ]:
RUN_NAME = 'asymmetric_payload_run'

# First run (nothing to resume from yet):
!python train.py \
  --num_envs 64 --device cuda --iterations 500 \
  --save_interval 100 \
  --wandb --run_name {RUN_NAME} \
  --log_dir {CHECKPOINT_DIR}/{RUN_NAME}


## Resuming after a disconnect

Colab free tier can drop the session mid-run without warning. Because `--log_dir` points at
Drive, whatever was already checkpointed (every `--save_interval` iterations) is safe. To
resume: reconnect, re-run the mount/clone/install/login cells above, then run the cell below
instead of the first training cell -- pick the highest `model_<N>.pt` actually present in your
Drive checkpoint folder.

**Note:** `--iterations` here means "how many *more* iterations to run from the checkpoint",
not an absolute target -- `runner.load()` restores the saved iteration count, and `learn()` adds
`--iterations` on top of that.

In [ ]:
RUN_NAME = 'asymmetric_payload_run'
# Runtime disconnected mid-run -- last checkpoint actually saved on Drive was iteration 2000.
# Resuming from there (not model_499.pt) so we don't lose the progress made since.
RESUME_FROM = f'{CHECKPOINT_DIR}/{RUN_NAME}/model_2000.pt'

# 2000 already done + 3000 more here = 5000 total (same overall target as before the disconnect).
!python train.py \
  --num_envs 64 --device cuda --iterations 3000 \
  --save_interval 250 \
  --wandb --run_name {RUN_NAME} \
  --log_dir {CHECKPOINT_DIR}/{RUN_NAME} \
  --resume {RESUME_FROM}


## Export the final policy to ONNX (also written straight to Drive)

In [ ]:
# NOTE: with --iterations 1500 (0-indexed loop), the LAST checkpoint is model_1499.pt, not
# model_1500.pt -- rsl_rl's runner saves every --save_interval iterations plus one final save
# at whatever iteration the loop actually stopped on. Check {CHECKPOINT_DIR}/{RUN_NAME}/ and use
# whichever model_<N>.pt is actually the highest N present.
!python export_policy.py \
  --checkpoint {CHECKPOINT_DIR}/{RUN_NAME}/model_1499.pt \
  --out {CHECKPOINT_DIR}/{RUN_NAME}/g1_policy.onnx